# 06 测试与对比
从下面列出的文件夹直接读取训练结果，在共同的时间测试队列上计算指标。某个实验还没有完成时跳过，已经完成的实验可以先比较。

阈值使用开发队列确定的值；本节不再训练模型或在测试集挑阈值。模型选择和调参先在开发队列完成，时间测试用于最后比较。


## 结果位置
默认目录对应 01、02、021、03、05 的输出。实验另存到其他目录时，只改下方相应路径。异地训练得到的文件放到这里即可，不需要清单、导入登记或版本检查。

021的原网络和融合结果显示为 `OriginalResNet`、`OriginalResNet_Clinical5`；临床单独模型共用 `Clinical5` 这一行。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.ndim != 5 or images.shape[1:3] != (4, 3) or masks.shape != images.shape or min(images.shape[-2:]) < 32:
        raise ValueError("分类输入应为[N,4,3,H,W]，影像与mask尺寸须一致，H和W至少为32。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
DL_MODELS = {
    "ResNet": RESULTS / "resnet",
    "OriginalResNet": RESULTS / "resnet_original",
    "T1_only": RESULTS / "ablation" / "t1",
    "T2_only": RESULTS / "ablation" / "t2",
    "DWI_only": RESULTS / "ablation" / "dwi",
    "No_mask": RESULTS / "ablation" / "no_mask",
    "No_attention": RESULTS / "ablation" / "no_attention",
}
RADIOMICS_DIR = RESULTS / "radiomics"
SEGMENTATION_DIR = RESULTS / "segmentation"
REPORT_DIR = RESULTS / "reports"
EXTRA_PREDICTION_FILES = []
RUN_TEST = False
N_BOOTSTRAP = 1000


## 读取模型并生成测试概率
影像模型读取 `model.pt`，临床融合读取 `clinical_models.joblib`，影像组学读取 `models.joblib` 和 `features.csv`。都按 MR 编号匹配患者。

如果存在 `validation_masks.npz`，还会用同一个分类模型把人工 mask 换成自动 mask，再计算一次分类结果。这个对照用于观察替换 mask 的影响，没有重新训练分类网络。

分类网络按权重中保存的设置进行输入标准化，避免漏做或重复做。缓存大小需与训练输入一致：本次重建为512后，尚未重新训练的224模型会提示并跳过；新02及其融合完成后可直接测试。

自动mask也需与当前分类缓存同尺寸。若仍是旧224输出，本节跳过自动mask分类，并提示到05使用已有分割权重重新导出。


In [ ]:
import random
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class GatedAttention(nn.Module):
    def __init__(self, features, hidden=128):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.score(torch.tanh(self.value(x)) * torch.sigmoid(self.gate(x)))


class MaskEncoder(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, features),
            nn.LayerNorm(features), nn.GELU(), nn.Dropout(.3))

    def forward(self, x):
        return self.net(x.float())


class ResNetClassifier(nn.Module):
    def __init__(self, image_dim=256, mask_dim=64, attention_dim=128,
                 use_mask=True, use_attention=True, pretrained=False, normalize_input=False):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_features.requires_grad_(False)
        self.image_features.eval()
        self.normalize_input = normalize_input
        self.register_buffer("input_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("input_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.image_projection = nn.Sequential(nn.Linear(2048, image_dim), nn.LayerNorm(image_dim), nn.GELU(), nn.Dropout(.3))
        self.mask_features = MaskEncoder(mask_dim) if use_mask else None
        width = image_dim + (mask_dim if use_mask else 0)
        self.attention = GatedAttention(width, attention_dim) if use_attention else None
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Dropout(.4), nn.Linear(width, 1))

    def train(self, mode=True):
        super().train(mode)
        # 后续模块照常训练，固定主干及其BatchNorm统计量。
        self.image_features.eval()
        return self

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image_input = images.reshape(b * slices, channels, h, w)
        if self.normalize_input:
            image_input = (image_input - self.input_mean) / self.input_std
        features = self.image_projection(self.image_features(image_input).flatten(1))
        features = features.reshape(b, slices, -1)
        if self.mask_features is not None:
            lesion = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
            features = torch.cat([features, lesion], dim=-1)
        if self.attention is None:
            pooled = features.mean(dim=1)
        else:
            weights = self.attention(features).softmax(dim=1)
            pooled = (features * weights).sum(dim=1)
        return self.head(pooled).squeeze(-1)


def sequence_input(images, sequence="all"):
    if sequence == "all":
        return images
    channel = SEQUENCES.index(sequence)
    return images[:, :, [channel, channel, channel]]


@torch.no_grad()
def predict(model, images, masks, batch_size=16):
    model.eval()
    probabilities = []
    for start in range(0, len(images), batch_size):
        x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        m = torch.as_tensor(masks[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x, m)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities)


def load_classifier(file):
    saved = torch.load(file, map_location="cpu", weights_only=True)
    if saved.get("architecture") == "original_resnet50":
        model = OriginalResNet(pretrained=False).to(DEVICE)
    else:
        model = ResNetClassifier(**saved["model_args"], pretrained=False).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), saved


## 021原网络的权重读取
下面是与021一致的原网络定义，用于读取 `results/resnet_original/model.pt`。两种结构由各自权重中的模型类型字段区分，读取时不会下载ImageNet权重。


In [ ]:
class OriginalMaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


class OriginalResNet(nn.Module):
    def __init__(self, pretrained=False) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = OriginalMaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss


def choose_threshold(labels, probabilities):
    fpr, tpr, thresholds = roc_curve(labels, probabilities)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def classification_metrics(labels, probabilities, threshold):
    labels, probabilities = np.asarray(labels, int), np.asarray(probabilities, float)
    if not np.isfinite(threshold):
        raise ValueError("分类阈值必须是开发队列确定的有限数值。")
    if set(labels) != {0, 1} or not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("指标需要两类结局，以及0～1之间的有效概率。")
    tn, fp, fn, tp = confusion_matrix(labels, probabilities >= threshold, labels=[0, 1]).ravel()
    return {"n": len(labels), "fn": int(fn), "fp": int(fp),
            "fnr": float(fn / (tp + fn)), "fpr": float(fp / (tn + fp)),
            "auc": float(roc_auc_score(labels, probabilities)), "threshold": float(threshold),
            "accuracy": float((tp + tn) / len(labels)), "sensitivity": float(tp / (tp + fn)),
            "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(labels, probabilities))}


In [ ]:
import joblib
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve


def collect_predictions(cases, images, masks, dl_models, radiomics_dir, segmentation_dir, extra_files):
    index = np.flatnonzero(cases.split.eq("validation"))
    test = cases.iloc[index].reset_index(drop=True)
    x, m = images[index], masks[index]
    if set(test.label) != {0, 1}:
        raise ValueError("测试队列需要同时包含两类结局。")
    tables, probabilities = [], {}

    def add(name, probability, threshold):
        if name in probabilities:
            raise ValueError(f"结果名称重复：{name}")
        classification_metrics(test.label, probability, threshold)
        frame = test[["mr", "label"]].copy()
        frame["model"], frame["probability"], frame["threshold"] = name, probability, float(threshold)
        tables.append(frame)
        probabilities[name] = np.asarray(probability)

    for name, folder in dl_models.items():
        file = folder / "model.pt"
        if not file.exists():
            print(f"跳过{name}：尚无{file.name}")
            continue
        model, saved = load_classifier(file)
        expected_size = tuple(saved.get("input_size", [224, 224]))
        if tuple(x.shape[-2:]) != expected_size:
            print(f"跳过{name}：权重使用{expected_size}，当前缓存为{x.shape[-2:]}；需使用同尺寸训练结果。")
            del model
            continue
        if set(test.mr) & set(saved["train_mrs"]):
            raise ValueError(f"{name}的训练患者出现在测试队列中。")
        add(name, predict(model, sequence_input(x, saved["sequence"]), m), saved["threshold"])
        del model

    resnet_dir = dl_models.get("ResNet")
    auto_file = segmentation_dir / "validation_masks.npz"
    if resnet_dir is not None and "ResNet" in probabilities and auto_file.exists():
        with np.load(auto_file, allow_pickle=False) as data:
            identifiers = data["mrs"].astype(str).tolist()
            if len(set(identifiers)) != len(identifiers) or set(identifiers) != set(test.mr):
                raise ValueError("自动勾画结果与测试队列的MR编号不一致。")
            lookup = {mr: i for i, mr in enumerate(identifiers)}
            auto_masks = data["masks"][[lookup[mr] for mr in test.mr]]
        if auto_masks.shape != m.shape:
            print(f"跳过自动mask分类：自动mask尺寸{auto_masks.shape}与当前输入{m.shape}不同，请用05重新导出。")
        elif not np.isin(auto_masks, [0, 1]).all():
            raise ValueError("自动mask应为与分类输入同尺寸的[N,4,3,H,W]的0/1数组。")
        else:
            model, saved = load_classifier(resnet_dir / "model.pt")
            add("ResNet_AutoMask", predict(model, sequence_input(x, saved["sequence"]), auto_masks), saved["threshold"])
            del model

    clinical = test[CLINICAL].to_numpy(dtype=float)
    for source in ("ResNet", "OriginalResNet"):
        folder = dl_models.get(source)
        if folder is None or not (folder / "clinical_models.joblib").exists():
            continue
        models = joblib.load(folder / "clinical_models.joblib")
        if "Clinical5" not in probabilities:
            saved = models["Clinical5"]
            add("Clinical5", saved["model"].predict_proba(clinical)[:, 1], saved["threshold"])
        fusion_name = source + "_Clinical5"
        saved = models[fusion_name]
        for image_name, name in ((source, fusion_name), (source + "_AutoMask", source + "_AutoMask_Clinical5")):
            if image_name in probabilities:
                fused = np.column_stack([probabilities[image_name], clinical])
                add(name, saved["model"].predict_proba(fused)[:, 1], saved["threshold"])

    if (radiomics_dir / "models.joblib").exists() and (radiomics_dir / "features.csv").exists():
        features = pd.read_csv(radiomics_dir / "features.csv", dtype={"mr": str}).set_index("mr")
        if not features.index.is_unique or not set(test.mr) <= set(features.index):
            raise ValueError("影像组学特征表缺少测试患者或包含重复MR。")
        values = test.set_index("mr").join(features, how="left").replace([np.inf, -np.inf], np.nan)
        for name, saved in joblib.load(radiomics_dir / "models.joblib").items():
            add(name, saved["model"].predict_proba(values[saved["features"]])[:, 1], saved["threshold"])

    for file in extra_files:
        previous = pd.read_csv(file, dtype={"mr": str})
        for name, group in previous.groupby("model", sort=False):
            group = group.set_index("mr")
            if not group.index.is_unique or set(group.index) != set(test.mr):
                raise ValueError(f"{file.name}的测试患者不同，不能直接比较。")
            group = group.loc[test.mr]
            if not np.array_equal(group.label, test.label) or group.threshold.nunique() != 1:
                raise ValueError(f"{file.name}的结局或阈值不一致。")
            add(str(name), group.probability.to_numpy(), float(group.threshold.iloc[0]))
    if not tables:
        raise FileNotFoundError("没有找到可测试的模型；请先完成训练，或修改本Notebook中的结果目录。")
    return pd.concat(tables, ignore_index=True)


def summarize_predictions(predictions, output, n_bootstrap=1000):
    output.mkdir(parents=True, exist_ok=True)
    groups = {name: table.sort_values("mr").reset_index(drop=True) for name, table in predictions.groupby("model", sort=False)}
    first = next(iter(groups.values()))
    y = first.label.to_numpy(dtype=int)
    rng = np.random.default_rng(SEED)
    indices = [rng.integers(0, len(y), len(y)) for _ in range(n_bootstrap)]
    indices = [index for index in indices if len(np.unique(y[index])) == 2]
    bootstrap, rows = {}, []
    for name, table in groups.items():
        probability = table.probability.to_numpy()
        bootstrap[name] = np.array([roc_auc_score(y[index], probability[index]) for index in indices])
        lower, upper = np.quantile(bootstrap[name], [.025, .975])
        rows.append({"model": name, **classification_metrics(y, probability, float(table.threshold.iloc[0])),
                     "auc_ci_lower": float(lower), "auc_ci_upper": float(upper)})
    metrics = pd.DataFrame(rows)
    predictions.to_csv(output / "validation_predictions.csv", index=False)
    metrics.to_csv(output / "validation_metrics.csv", index=False)
    comparisons = []
    if "ResNet" in groups:
        baseline_auc = float(roc_auc_score(y, groups["ResNet"].probability))
        for name in groups:
            if name == "ResNet":
                continue
            lower, upper = np.quantile(bootstrap[name] - bootstrap["ResNet"], [.025, .975])
            comparisons.append({"model": name, "reference": "ResNet",
                "auc_difference": float(roc_auc_score(y, groups[name].probability) - baseline_auc),
                "ci_lower": float(lower), "ci_upper": float(upper)})
    comparison_table = pd.DataFrame(comparisons, columns=["model", "reference", "auc_difference", "ci_lower", "ci_upper"])
    comparison_table.to_csv(output / "auc_comparisons.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for name, table in groups.items():
        fpr, tpr, _ = roc_curve(y, table.probability)
        axes[0].plot(fpr, tpr, label=f"{name}: {roc_auc_score(y, table.probability):.3f}")
        observed, predicted = calibration_curve(y, table.probability, n_bins=5, strategy="quantile")
        axes[1].plot(predicted, observed, "o-", label=name)
    for ax in axes:
        ax.plot([0, 1], [0, 1], "k--", alpha=.4)
        ax.legend(fontsize=7)
    axes[0].set(xlabel="False positive rate", ylabel="True positive rate", title="Temporal test ROC")
    axes[1].set(xlabel="Predicted probability", ylabel="Observed proportion", title="Calibration")
    fig.tight_layout()
    fig.savefig(output / "roc_calibration.png", dpi=160)
    plt.show()
    return metrics, comparison_table


## 计算指标与绘图
优先查看 `fn`（漏判人数）、`fp`（误报人数）、`fnr`（漏判率）和 `fpr`（误报率）；错误率分别以阳性、阴性人数为分母。仍输出AUC及其患者级bootstrap 95%区间、准确率、敏感度、特异度和Brier分数。另用相同的重采样患者计算各模型相对 ResNet 的 AUC 差值区间，并画 ROC 和校准图。

如需加入旧实验，把它的测试预测表路径放进 `EXTRA_PREDICTION_FILES`。CSV 每行是一位患者的一个模型结果，列名为 `mr,label,model,probability,threshold`；`model` 使用不同名称，`threshold` 填当时开发队列确定的阈值。开发集 OOF 不能作为测试预测加入。

这里直接使用各模型在开发阶段保存的阈值，不根据测试集的FN或FP重新选择阈值。回退02训练版本后，用相应的新权重和临床融合模型计算结果。


In [ ]:
if RUN_TEST:
    cases, images, masks = read_classification_data()
    predictions = collect_predictions(cases, images, masks, DL_MODELS, RADIOMICS_DIR,
                                      SEGMENTATION_DIR, EXTRA_PREDICTION_FILES)
    metrics, comparisons = summarize_predictions(predictions, REPORT_DIR, n_bootstrap=N_BOOTSTRAP)
    display(metrics.round(4))
    display(comparisons.round(4))
    dice_file = SEGMENTATION_DIR / "validation_dice.csv"
    if dice_file.exists():
        dice = pd.read_csv(dice_file, dtype={"mr": str})
        if set(dice.mr) != set(cases.loc[cases.split == "validation", "mr"]) or dice.duplicated(["mr", "label"]).any():
            raise ValueError("分割Dice表中的测试患者不同或有重复记录。")
        dice_summary = dice.groupby("label").agg(n=("mr", "size"), n_defined=("dice", "count"), mean_dice=("dice", "mean"))
        dice_summary.to_csv(REPORT_DIR / "segmentation_metrics.csv")
        display(dice_summary)
else:
    print("确认结果路径后，将RUN_TEST改为True。")


结果保存在 `results/reports`：`validation_predictions.csv` 是全部患者预测，`validation_metrics.csv` 是分类指标，`auc_comparisons.csv` 是 AUC 差值，`roc_calibration.png` 是对比图。有分割结果时另存 `segmentation_metrics.csv`。同一路径重复运行会更新这些报告。
